# Swaption Backtest (Conceal Last 6 Rows)
This notebook hides the final 6 rows, trains on earlier history only, predicts those hidden rows recursively, and compares predictions vs truth.


In [2]:
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option("display.max_columns", 12)


In [4]:
# Load local training data (already in repo)
df = pd.read_parquet("../data/level1.parquet").copy()
df["Date"] = pd.to_datetime(df["Date"], format="mixed", dayfirst=True)
df = df.sort_values("Date").reset_index(drop=True)

feature_cols = [c for c in df.columns if c != "Date"]
df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors="coerce")

after_na = df[feature_cols].isna().sum().sum()
if after_na:
    # Forward/backward fill handles any sparse missing entries for modeling.
    df[feature_cols] = df[feature_cols].ffill().bfill()

print("Rows:", len(df), "Features:", len(feature_cols))
print("Date range:", df["Date"].min(), "to", df["Date"].max())
print("Remaining NA:", int(df[feature_cols].isna().sum().sum()))


Rows: 494 Features: 224
Date range: 2050-01-01 00:00:00 to 2051-12-23 00:00:00
Remaining NA: 0


In [5]:
# Convert series into supervised learning format using a rolling lookback window.
# X_t uses the previous `lookback` rows to predict row t.
lookback = 12

def make_supervised(dataframe, feature_names, lookback_steps=12):
    values = dataframe[feature_names].values.astype(np.float64)
    dates = dataframe["Date"].values
    X, y, y_dates = [], [], []
    for t in range(lookback_steps, len(dataframe)):
        X.append(values[t - lookback_steps:t].reshape(-1))
        y.append(values[t])
        y_dates.append(dates[t])
    return np.array(X), np.array(y), np.array(y_dates)

X_all, y_all, y_dates = make_supervised(df, feature_cols, lookback_steps=lookback)
print("Supervised X shape:", X_all.shape)
print("Supervised y shape:", y_all.shape)


Supervised X shape: (482, 2688)
Supervised y shape: (482, 224)


In [6]:
# Backtest setup: conceal the final 6 rows as pseudo-test
horizon = 6
hide_start = len(df) - horizon

if hide_start <= lookback:
    raise ValueError("Not enough rows for lookback + concealed horizon.")

train_df = df.iloc[:hide_start].copy()
hidden_df = df.iloc[hide_start:].copy()  # kept for evaluation only

# Build supervised training samples only from the visible training window
X_train, y_train, dates_train = make_supervised(train_df, feature_cols, lookback_steps=lookback)

print("Visible training rows:", len(train_df))
print("Hidden rows:", len(hidden_df))
print("Training samples:", len(X_train))
print("Train end date:", train_df["Date"].iloc[-1])
print("Hidden date range:", hidden_df["Date"].iloc[0], "to", hidden_df["Date"].iloc[-1])


Train samples: 385 Test samples: 97
Train end date: 2051-08-03 00:00:00
Test start date: 2051-08-04 00:00:00


In [7]:
# Model: scale features + regularized linear regression for multi-output targets
model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=2.0, random_state=42))
])

model.fit(X_train, y_train)
print("Model trained on visible data only.")


Test MAE : 0.00284945
Test RMSE: 0.00375118


In [8]:
# Recursively predict the concealed final 6 rows
history = train_df[feature_cols].values.astype(np.float64).copy()
concealed_preds = []

for _ in range(horizon):
    x_next = history[-lookback:].reshape(1, -1)
    y_next = model.predict(x_next)[0]
    concealed_preds.append(y_next)
    history = np.vstack([history, y_next])

concealed_preds = np.array(concealed_preds)

y_true_hidden = hidden_df[feature_cols].values.astype(np.float64)
mae_hidden = mean_absolute_error(y_true_hidden, concealed_preds)
rmse_hidden = np.sqrt(mean_squared_error(y_true_hidden, concealed_preds))

print(f"Concealed-6 MAE : {mae_hidden:.8f}")
print(f"Concealed-6 RMSE: {rmse_hidden:.8f}")

# Row-wise error summary for the 6 concealed steps
row_mae = np.mean(np.abs(y_true_hidden - concealed_preds), axis=1)
comparison_summary = pd.DataFrame({
    "Date": hidden_df["Date"].values,
    "Row_MAE": row_mae,
})
comparison_summary


Forecast shape: (6, 225)


,Date,Tenor : 10; Maturity : 0.0833333333333333,Tenor : 10; Maturity : 0.25,Tenor : 10; Maturity : 0.5,Tenor : 10; Maturity : 0.75,Tenor : 10; Maturity : 1,...,Tenor : 9; Maturity : 25,Tenor : 9; Maturity : 3,Tenor : 9; Maturity : 30,Tenor : 9; Maturity : 4,Tenor : 9; Maturity : 5,Tenor : 9; Maturity : 7
0,2051-12-25,0.034984,0.062390,0.086908,0.107011,0.126025,...,0.323849,0.201751,0.344477,0.217267,0.234291,0.251316
1,2051-12-27,0.035500,0.063154,0.087833,0.108030,0.127024,...,0.326429,0.203202,0.348031,0.218799,0.235793,0.252859
2,2051-12-29,0.036066,0.063934,0.088712,0.108953,0.127832,...,0.328016,0.204214,0.350435,0.219835,0.236774,0.253949
3,2051-12-31,0.036520,0.064575,0.089448,0.109733,0.128533,...,0.329424,0.205131,0.352452,0.220778,0.237666,0.254911
4,2052-01-02,0.037092,0.065444,0.090513,0.110907,0.129730,...,0.331308,0.206738,0.354997,0.222376,0.239191,0.256314
5,2052-01-04,0.037903,0.066684,0.092035,0.112581,0.131501,...,0.333117,0.208951,0.357559,0.224461,0.241140,0.257951


In [10]:
# Build full comparison tables and save
pred_hidden_df = pd.DataFrame(concealed_preds, columns=feature_cols)
pred_hidden_df.insert(0, "Date", hidden_df["Date"].values)

actual_hidden_df = hidden_df[["Date"] + feature_cols].reset_index(drop=True)

# Long-form comparison for easy inspection
comparison_long = (
    actual_hidden_df.set_index("Date").stack().rename("actual").to_frame()
    .join(pred_hidden_df.set_index("Date").stack().rename("pred"))
    .reset_index()
    .rename(columns={"level_1": "instrument"})
)
comparison_long["abs_err"] = (comparison_long["actual"] - comparison_long["pred"]).abs()

pred_hidden_df.to_parquet("../data/level1_last6_pred.parquet", index=False)
comparison_long.to_parquet("../data/level1_last6_comparison_long.parquet", index=False)
print("Saved:")
print("- ../data/level1_last6_pred.parquet")
print("- ../data/level1_last6_comparison_long.parquet")

comparison_long.head(10)


Saved forecast to data/level1_next6_forecast.parquet
